<a href="https://colab.research.google.com/github/johramoosa/single-cell-ml-analysis/blob/main/single_cell_RNA_seq_ML_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Single-Cell RNA-seq Machine Learning Analysis

This project explores single-cell RNA sequencing (scRNA-seq) data using Python and modern data science tools.

Single-cell RNA-seq measures gene expression for thousands of individual cells, making it possible to identify distinct cell populations and study cellular heterogeneity at high resolution.

This analysis uses a public immune cell dataset and combines **unsupervised learning** techniques, such as clustering and dimensionality reduction, with a later **supervised machine learning** step for cell type classification.

The project is designed to demonstrate a practical end-to-end workflow for analyzing high-dimensional biological data and applying machine learning methods in genomics.

---

## Tools

- Python
- Google Colab
- Scanpy
- Pandas
- NumPy
- Scikit-learn
- Matplotlib

---

## Dataset

**PBMC (Peripheral Blood Mononuclear Cells) single-cell RNA-seq dataset**

This dataset contains gene expression measurements for thousands of individual immune cells.

- Each **row** represents a cell
- Each **column** represents a gene

This forms a high-dimensional expression matrix of:

**cells × genes**

After quality control and highly variable gene selection, the working dataset was reduced to a smaller, more informative feature space for downstream analysis.

---

## Analysis Workflow

The notebook follows a standard single-cell RNA-seq analysis pipeline with an additional machine learning component.

1. **Load dataset**  
   Import the PBMC single-cell dataset and inspect its structure.

2. **Initial exploration and quality assessment**  
   Examine total counts, number of detected genes per cell, and highly expressed genes.

3. **Quality control and filtering**  
   Remove low-quality cells and low-information genes. Evaluate mitochondrial gene percentage and filtering thresholds.

4. **Normalization and log transformation**  
   Normalize expression counts across cells and apply log transformation to stabilize variance.

5. **Highly variable gene selection**  
   Select the most informative genes for downstream analysis.

6. **Scaling**  
   Standardize the selected gene expression matrix before dimensionality reduction.

7. **Dimensionality reduction (PCA)**  
   Reduce the dataset into principal components and interpret major axes of variation.

8. **Neighbor graph construction**  
   Build a graph of transcriptionally similar cells using PCA space.

9. **UMAP visualization**  
   Project cells into 2D space for visualization of global and local population structure.

10. **Leiden clustering**  
    Identify groups of cells with similar gene expression profiles.

11. **Cell type classification (Machine Learning)**  
    Train a supervised model to predict cell types based on expression-derived features.

12. **Feature importance analysis**  
    Identify genes or features contributing most strongly to classification performance.

---

## Project Goals

- Understand the structure of single-cell RNA-seq datasets
- Perform a standard preprocessing and analysis workflow for scRNA-seq data
- Apply dimensionality reduction and clustering to discover cell populations
- Build a supervised machine learning model for cell type classification
- Interpret biologically meaningful patterns from gene expression data

---

## Progress So Far

The following stages have been completed:

- Dataset loading and structure inspection
- Basic quality-control metric calculation
- Visualization of total counts and detected genes per cell
- Exploration of highly expressed genes
- Cell and gene filtering
- Mitochondrial percentage calculation and QC assessment
- Normalization and log transformation
- Highly variable gene selection
- Scaling
- PCA computation and interpretation
- Neighbor graph construction
- UMAP embedding
- Leiden clustering
- Identified marker genes for each cluster
- Assigned biological cell-type labels

### Next Steps

- Build a supervised classification model
- Evaluate feature importance for interpretable machine learning

In [ ]:
!pip install scanpy
!pip install leidenalg

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
#load data
adata = sc.datasets.pbmc3k()

adata

In [ ]:
adata.shape

The dataset contains 2,700 cells and 32,738 genes before preprocessing.
This high-dimensional structure is typical of single-cell RNA-seq data and motivates downstream filtering and dimensionality reduction.


In [ ]:
#adata.var → gene metadata
adata.var.head()

In [ ]:
#adata.X → gene expression matrix
adata.X

In [ ]:
#adata.X → gene expression matrix
#adata.obs → cell metadat
#Calculate total RNA detected in each cell
#New feature created. total_counts = sum of RNA counts across all genes in each cell

adata.obs["total_counts"] = adata.X.sum(axis=1)
adata.obs.head()

In [ ]:
plt.hist(adata.obs["total_counts"], bins=50)
plt.xlabel("Total RNA counts per cell") #sum of gene counts
plt.ylabel("Number of cells")
plt.title("Distribution of RNA counts per cell")
plt.show()

In [ ]:
print("Minimum counts:", adata.obs["total_counts"].min())
print("Median counts:", adata.obs["total_counts"].median())
print("Maximum counts:", adata.obs["total_counts"].max())

#Analysis
The distribution of RNA counts per cell shows that most cells contain around ~2000 detected transcripts, with a small number of cells having higher counts. This distribution is typical for PBMC single-cell datasets and helps guide quality control filtering.

Numbers look healthy. Dataset is clean enough to proceed.

In [ ]:
#Calculate Genes per Cell. This measures gene diversity per cell.
#Created new feature. n_genes = number of genes with expression > 0 in each cell

adata.obs["n_genes"] = (adata.X > 0).sum(axis=1)#number of genes expressed (i.e., that have non zero values)
adata.obs.head()

In [ ]:
plt.hist(adata.obs["n_genes"], bins=50)
plt.xlabel("Number of genes detected per cell")
plt.ylabel("Number of cells")
plt.title("Genes detected per cell")
plt.show()


In [ ]:
print("Minimum genes:", adata.obs["n_genes"].min())
print("Median genes:", adata.obs["n_genes"].median())
print("Maximum genes:", adata.obs["n_genes"].max())

In [ ]:
plt.scatter(adata.obs["total_counts"], adata.obs["n_genes"], alpha=0.3)
plt.xlabel("Total RNA counts")
plt.ylabel("Genes detected")
plt.title("Genes vs RNA counts per cell")
plt.show()

#Analysis
Dataset behaves normally showing a diagonal trend.
Which means a clear relationship: more RNA → more genes detected.

Two common quality control metrics in single-cell RNA-seq are:

- Total RNA counts per cell
- Number of genes detected per cell

Total counts measure the overall sequencing depth for a cell, while the number of genes detected measures the diversity of expressed genes.

Healthy cells typically show a positive relationship between these two metrics.

In [ ]:
#Top 20 Most Expressed Genes Across Cells
#This visualization helps detect dominating genes.
#Showing balanced gene expression across cells.

ax = sc.pl.highest_expr_genes(adata, n_top=20, show=False)
ax.set_title("Top 20 Most Expressed Genes Across Cells")
plt.show()

#Analysis
The distribution of the most expressed genes was inspected to ensure that no single gene dominated the dataset. The top genes contributed only a small fraction of total counts, suggesting balanced gene expression across cells.

The most highly expressed genes in the dataset include MALAT1 and TMSB4X, which are commonly observed among highly expressed transcripts in immune cell datasets. Each top gene contributes only a small fraction of total counts (~2–3%), indicating that no single gene dominates the dataset.

## Preprocessing

In this stage, low-quality genes and cells were filtered, mitochondrial RNA content was calculated, and the dataset was normalized for downstream analysis.

### Steps performed
1. Removed genes expressed in fewer than 3 cells
2. Removed cells with fewer than 200 detected genes
3. Calculated mitochondrial gene percentage per cell
4. Filtered cells with high mitochondrial RNA content
5. Normalized total counts per cell
6. Applied log transformation
7. Applied scaling

This preprocessing step reduces noise and makes cells more comparable for dimensionality reduction and clustering.

In [ ]:
adata_raw = adata.copy()

#Filter low-quality genes
Remove genes detected in very few cells. (get rids of columns)

In [ ]:
#Remove extremely rare genes
#Keeping genes expressed in at least 3 cells

sc.pp.filter_genes(adata, min_cells=3)

In [ ]:
adata.shape

#Filter low-quality cells
Remove cells with too few detected genes. (get rids of rows)

In [ ]:
#Keep cells with at least 200 detected genes
#Removing likely low-quality or empty droplets

sc.pp.filter_cells(adata, min_genes=200)

In [ ]:
#Minimum genes: 212, so nothing filtered out
adata.shape

#Add mitochondrial gene percentage

In [ ]:
adata.var["mt"] = adata.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True)
adata.obs.head()

#adds columns such as total_counts_mt, pct_counts_mt
#pct_counts_mt means what percent of a cell’s RNA comes from mitochondrial genes (High mitochondrial percentage can indicate stressed or damaged cells.)


#Summary Statistics

In [ ]:
adata.obs[["total_counts", "n_genes_by_counts", "pct_counts_mt"]].describe()

#Filter cells with high mitochondrial percentage

In [ ]:
#Filter cells with high mitochondrial percentage (getting rid of rows)
#Keeping cells with less than 5% mitochondrial RNA
adata = adata[adata.obs["pct_counts_mt"] < 5].copy()

In [ ]:
adata.shape

#Normalize counts
Rescale each cell so total counts become about 10,000.

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)

#Log transform
Compress large values and makes the data easier to analyze.

In [ ]:
sc.pp.log1p(adata)

In [ ]:
#Saving in a layer
adata.layers["log_normalized"] = adata.X.copy()

In [ ]:
adata

In [ ]:
#Current Stat
print(adata.shape)
print(adata.obs[["total_counts", "n_genes_by_counts", "pct_counts_mt"]].describe())
print(adata.obs["pct_counts_mt"].max())

#Identify Highly Variable Genes (HVGs)

In [ ]:
#selects the top 2000 most variable genes
sc.pp.highly_variable_genes(adata, n_top_genes=2000)

In [ ]:
adata.var.head()

In [ ]:
adata.var["highly_variable"].sum()

In [ ]:
#Genes above the expected variance trend are selected.
sc.pl.highly_variable_genes(adata)

In [ ]:
#HVG only data
adata = adata[:, adata.var["highly_variable"]].copy()

In [ ]:
adata.shape

## Scaling

After selecting highly variable genes, the data was scaled so that each gene had comparable magnitude across cells. This helps prevent highly expressed genes from dominating principal component analysis (PCA). Values were clipped at a maximum of 10 to reduce the effect of extreme outliers.

In [ ]:
#Scaling
#different genes still have different ranges so scaling to make roughly mean 0, std 1. which helps the later analysis
sc.pp.scale(adata, max_value=10)

## Principal Component Analysis (PCA)

After preprocessing and selecting highly variable genes, principal component analysis (PCA) was applied to reduce the dimensionality of the dataset. PCs correspond to major biological axes of variation.

PCA summarizes variation across thousands of genes into a smaller number of components, making it easier to visualize the data and prepare for downstream clustering.

The first few principal components capture the strongest sources of variation across cells, which often correspond to biological differences between cell populations.

PCA acts as a denoising compression step, before we perform advanced clustering.

In [ ]:
#Following features are computed and added
#adata.obsm["X_pca"] → PCA coordinates for each cell
#adata.varm["PCs"] → contribution of each gene to each PC
#adata.uns["pca"] → extra PCA info

sc.tl.pca(adata, svd_solver="arpack")

In [ ]:
adata.obsm["X_pca"]

In [ ]:
adata.obsm["X_pca"].shape

In [ ]:
#PCA variance ratio plot
#This plot shows: How much variance each principal component explains
sc.pl.pca_variance_ratio(adata, log=True)

In [ ]:
#PCA scatter plot
sc.pl.pca(adata)

In [ ]:
#Gene contribution to PC. what biological signals PCA discovered
#This plot shows: Which genes are responsible for the variation captured by each principal component?
#PCA loading plots reveal which genes contribute most strongly to each principal component, helping interpret the biological signals driving cell separation.
sc.pl.pca_loadings(adata)

#Analysis
PCA loading plots reveal which genes contribute most strongly to each principal component, helping interpret the biological signals driving cell separation.

Based on the genes in loading plot,

1. PC1 likely separates: myeloid cells  VS. T cells

2. PC2 separating: cytotoxic lymphocytes VS. antigen presenting cells

In blood immune cells, the biggest differences are typically:

1. lineage differences
2. cytotoxic activity
3. ntigen presentation

So it makes sense that PC1 and PC2 capture those.
This means PCA is already identifying major immune programs in the dataset

PCA loading analysis revealed biologically meaningful gene programs. PC1 appeared to separate myeloid lineage cells from T lymphocytes, while PC2 highlighted cytotoxic immune signatures (NKG7, GZMB, PRF1) contrasted with antigen presentation genes (HLA-DPB1, HLA-DRB1).

A principal component often represents a gene program, meaning a group of genes that tend to go up and down together across cells.
In single-cell data, genes are often not acting independently. Instead, sets of genes reflect:
1. a cell type
2. an activation state
3. a biological pathway
4. a stress response

## Neighbor Graph, UMAP, and Clustering

### Objective
Construct a graph-based representation of cell similarity, visualize the dataset in low dimensions, and identify transcriptionally similar cell groups.

### Steps

#### 1. Neighbor graph construction
A nearest-neighbor graph was built using the PCA representation of the dataset.

- Input used: principal components
- Example parameters:
  - `n_neighbors = 10`
  - `n_pcs = 20`

This step identifies, for each cell, the most similar cells in reduced-dimensional space.

#### 2. UMAP embedding
A UMAP embedding was computed from the neighbor graph.

UMAP projects the high-dimensional similarity structure into a 2D space so that:
- cells with similar expression profiles appear close together
- dissimilar cells appear farther apart

UMAP is used primarily for visualization.

#### 3. Leiden clustering
Leiden clustering was applied to the same neighbor graph.

This step identifies groups of tightly connected cells in the graph and assigns cluster labels.

UMAP and Leiden use the same neighborhood structure, but their purposes differ:
- **UMAP** provides a visual layout
- **Leiden** defines cluster membership


In [ ]:
# Step 1: build the neighbor graph
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=20)

In [ ]:
# Step 2: compute UMAP
sc.tl.umap(adata)

In [ ]:
# Step 3: perform clustering
sc.tl.leiden(adata, resolution=0.5)

In [ ]:
# Visualize UMAP colored by cluster
sc.pl.umap(adata, color="leiden")

In [ ]:
adata.obs["leiden"].value_counts().sort_index()

#Analysis
### Results

The UMAP embedding showed several clearly separated cell groups, with some clusters closely adjacent but not fully overlapping.

Leiden clustering identified **9 clusters**:

- Cluster 0: 1173 cells
- Cluster 1: 463 cells
- Cluster 2: 343 cells
- Cluster 3: 273 cells
- Cluster 4: 177 cells
- Cluster 5: 158 cells
- Cluster 6: 36 cells
- Cluster 7: 12 cells
- Cluster 8: 8 cells

### Interpretation

The presence of several major clusters suggests broad immune cell populations, while the very small clusters may correspond to rare cell types, transitional states, or possible technical outliers.

At this stage, cluster identities remain unlabeled. Biological interpretation will require marker-gene analysis.

### Progress Status

Completed:
- Quality control and filtering
- Normalization and log transformation
- Highly variable gene selection
- Scaling
- PCA
- Neighbor graph construction
- UMAP embedding
- Leiden clustering

### Next Step
Identify marker genes for each cluster and begin cell-type annotation.

#Find marker genes

In [ ]:
adata.X = adata.layers["log_normalized"].copy()
sc.tl.rank_genes_groups(adata, groupby="leiden", method="wilcoxon")#note marker ranking done possibly on HVGs and scaled values. NEED TO CHECK
#For each Leiden cluster, Scanpy compares that cluster against all other cells and finds genes that are more strongly expressed in that cluster compared to all other clusters.

#So it is basically asking:

#what genes make cluster 0 special?
#what genes make cluster 1 special?
#what genes make cluster 2 special?... so on

In [ ]:
#Visualize top marker genes

In [ ]:
sc.pl.rank_genes_groups(adata, n_genes=10, sharey=False)

#Get marker names

In [ ]:
sc.get.rank_genes_groups_df(adata, group=None).head(20)

#print top genes as text

In [ ]:
result = adata.uns["rank_genes_groups"]
groups = result["names"].dtype.names

for group in groups:
    print(f"\nCluster {group}")
    print(result["names"][group][:10])


##Analysis
| Cluster | Likely identity                   | Why                                      |
| ------- | --------------------------------- | ---------------------------------------- |
| 0       | T cells                           | `LTB`, `IL32`                            |
| 1       | Classical monocytes               | `LYZ`, `S100A8`, `S100A9`, `FCN1`        |
| 2       | B cells                           | `CD79A`, `CD79B`, `MS4A1`, `CD74`        |
| 3       | Cytotoxic lymphocytes             | `CCL5`, `NKG7`, `CST7`, `GZMA`, `GZMK`   |
| 4       | FCGR3A+ / non-classical monocytes | `LST1`, `FCER1G`, `FCGR3A`               |
| 5       | NK cells                          | `NKG7`, `GNLY`, `GZMB`, `PRF1`, `FGFBP2` |
| 6       | Dendritic cells                   | `FCER1A`, `CD74`, HLA-II genes           |
| 7       | Platelets                         | `PF4`, `PPBP`, `GNG11`                   |
| 8       | Cycling / proliferating cells     | `PCNA`, `BIRC5`, `ZWINT`, `STMN1`        |


##  Marker Gene Analysis and Preliminary Cell-Type Annotation

### Objective
Identify marker genes for each Leiden cluster and use canonical immune-cell markers to assign preliminary biological labels.

### Marker Gene Ranking
Marker genes were identified for each Leiden cluster using the Wilcoxon rank-sum test.

To ensure biologically meaningful marker detection, the analysis was performed on the saved **log-normalized expression matrix** rather than the scaled matrix used for PCA.

### Result
The top-ranked genes remained consistent after rerunning marker analysis on the log-normalized data, indicating that the clustering structure and marker patterns are robust.

### Preliminary Cluster Annotations

| Cluster | Representative marker genes | Preliminary annotation |
|---|---|---|
| 0 | `LTB`, `IL32`, `JUNB` | T cells |
| 1 | `LYZ`, `S100A8`, `S100A9`, `FCN1` | Classical monocytes |
| 2 | `CD74`, `CD79A`, `CD79B`, `MS4A1` | B cells |
| 3 | `CCL5`, `NKG7`, `CST7`, `GZMA`, `GZMK` | Cytotoxic lymphocytes |
| 4 | `LST1`, `FCER1G`, `AIF1`, `FCGR3A` | FCGR3A+ / non-classical monocytes |
| 5 | `NKG7`, `GNLY`, `GZMB`, `PRF1`, `FGFBP2` | NK cells |
| 6 | `HLA-DPA1`, `HLA-DPB1`, `CD74`, `FCER1A` | Dendritic cells |
| 7 | `PF4`, `PPBP`, `GNG11`, `SPARC` | Platelets |
| 8 | `PCNA`, `BIRC5`, `ZWINT`, `STMN1` | Cycling / proliferating cells |

### Interpretation
The analysis recovered several major PBMC populations, including lymphoid and myeloid cell types, as well as smaller rare populations such as dendritic cells, platelets, and proliferating cells.

The stability of the marker ranking after switching from scaled data to log-normalized data supports the reliability of these preliminary annotations.

### Next Step
Visualize canonical marker genes on the UMAP and assign finalized cell-type labels to each Leiden cluster.


## Marker Validation and Cell-Type Annotation

### Objective
Validate preliminary cluster annotations using canonical marker-gene expression on the UMAP and assign biological cell-type labels to Leiden clusters.

In [ ]:
# Visualize UMAP colored by cluster
sc.pl.umap(adata, color="leiden")

In [ ]:
sc.pl.umap(
    adata,
    color=["LTB", "LYZ", "MS4A1", "NKG7", "FCGR3A", "FCER1A", "PF4", "PCNA"],
    cmap="viridis"
)

#LTB 0, 2
#LYZ 1, 4, 6, a little 7
#MS4A1 2
#NKG7 3, 5, a little 8
#FCGR3A 4, 5, a little 3
#FCER1A 6
#PF4 7
#PCNA 8

In [ ]:
[c for c in ["CD3D", "CD3E", "TRAC", "TRBC1", "IL7R", "LTB", "IL32"] if c in adata.var_names]

In [ ]:
#For a cleaner T-cell confirmation, LTB is not as cleanly exclusive as something like CD3D for T cells
sc.pl.umap(adata, color=["IL32"], cmap="viridis")
#IL32 0, 3, 5

`NKG7` showed strong expression in clusters 3 and 5, supporting the presence of cytotoxic immune populations in both clusters. Cluster 5 remained consistent with NK cells based on the additional presence of `GNLY`, `GZMB`, `PRF1`, and `FGFBP2`, while cluster 3 was retained as a broader cytotoxic lymphocyte population.

Additional marker visualization showed that `LYZ` was enriched in clusters 1, 4, and 6, supporting their interpretation as monocyte and antigen-presenting myeloid populations. `FCGR3A` was strongest in cluster 4 and also present in cluster 5, consistent with FCGR3A+ monocytes and CD16+ NK-like cells, respectively.

Because `LTB` was not fully specific to the T-cell cluster, more canonical T-cell markers such as `CD3D`, `CD3E`, `IL32`, and `TRAC` were identified as better candidates for confirming the T-cell annotation. Only `IL32` was present.

`IL32` showed expression across clusters 0, 3, and 5, indicating that it is not exclusive to the T-cell cluster in this dataset. This pattern is consistent with broader lymphoid and cytotoxic immune expression. Therefore, the annotation of cluster 0 as T cells was based on the overall marker profile and exclusion of alternative immune identities, rather than on `IL32` alone.

In [ ]:
cluster_to_celltype = {
    "0": "T cells",
    "1": "Classical monocytes",
    "2": "B cells",
    "3": "Cytotoxic lymphocytes",
    "4": "FCGR3A+ monocytes",
    "5": "NK cells",
    "6": "Dendritic cells",
    "7": "Platelets",
    "8": "Cycling cells"
}

adata.obs["cell_type"] = adata.obs["leiden"].map(cluster_to_celltype)

In [ ]:
adata.obs["cell_type"].value_counts()

In [ ]:
sc.pl.umap(adata, color="cell_type")

### Marker Validation Summary

Canonical marker genes were visualized on the UMAP to confirm whether expression patterns matched the previously inferred cluster identities.

Observed marker patterns:

- `LTB` showed expression in clusters 0 and 2
- `IL32` showed expression in clusters 0, 3, and 5
- `LYZ` showed expression in clusters 1, 4, and 6, with a small signal in cluster 7
- `MS4A1` showed strongest expression in cluster 2
- `NKG7` showed strong expression in clusters 3 and 5, with a small signal in cluster 8
- `FCGR3A` showed strongest expression in cluster 4, with additional expression in cluster 5 and a small signal in cluster 3
- `FCER1A` showed strongest expression in cluster 6
- `PF4` showed strongest expression in cluster 7
- `PCNA` showed strongest expression in cluster 8

### Interpretation

The marker-gene UMAP plots were broadly consistent with the cluster-specific marker analysis and supported the biological interpretation of the Leiden clusters.

- Cluster 0 remained consistent with **T cells**
- Cluster 1 remained consistent with **classical monocytes**
- Cluster 2 was strongly confirmed as **B cells** by `MS4A1`
- Cluster 3 remained consistent with **cytotoxic lymphocytes**
- Cluster 4 was confirmed as **FCGR3A+ monocytes**
- Cluster 5 remained consistent with **NK cells**
- Cluster 6 was confirmed as **dendritic cells** by `FCER1A`
- Cluster 7 was confirmed as **platelets** by `PF4`
- Cluster 8 was confirmed as **cycling / proliferating cells** by `PCNA`

### Annotation Caveats

Some marker overlap across related immune populations was observed, which is biologically expected in PBMC datasets.

- `NKG7` was shared between clusters 3 and 5, supporting cytotoxic immune identity in both populations. Cluster 5 was retained as **NK cells** because of the additional strong expression of `GNLY`, `GZMB`, `PRF1`, and `FGFBP2`, while cluster 3 was retained as a broader **cytotoxic lymphocyte** population.
- `LYZ` was enriched in clusters 1, 4, and 6, supporting their interpretation as monocyte and antigen-presenting myeloid populations.
- `FCGR3A` was strongest in cluster 4 and also present in cluster 5, consistent with **FCGR3A+ monocytes** and **CD16+ NK-like cells**, respectively.
- Canonical T-cell markers such as `CD3D`, `CD3E`, and `TRAC` were not present in the HVG-filtered object. Therefore, cluster 0 was annotated as **T cells** based on the available lymphoid-associated markers `LTB` and `IL32`, together with the overall cluster profile and exclusion of alternative immune identities.
- `IL32` was not exclusive to cluster 0 and appeared across clusters 0, 3, and 5, indicating broader lymphoid and cytotoxic immune expression.


### Final Cluster Annotations

| Leiden cluster | Cell type |
|---|---|
| 0 | T cells |
| 1 | Classical monocytes |
| 2 | B cells |
| 3 | Cytotoxic lymphocytes |
| 4 | FCGR3A+ monocytes |
| 5 | NK cells |
| 6 | Dendritic cells |
| 7 | Platelets |
| 8 | Cycling cells |

### Final Cell-Type Counts

| Cell type | Number of cells |
|---|---:|
| T cells | 1173 |
| Classical monocytes | 463 |
| B cells | 343 |
| Cytotoxic lymphocytes | 273 |
| FCGR3A+ monocytes | 177 |
| NK cells | 158 |
| Dendritic cells | 36 |
| Platelets | 12 |
| Cycling cells | 8 |

### Outcome
Leiden clusters were successfully mapped to interpretable biological labels and stored in `adata.obs["cell_type"]` for downstream analysis.

### Next Step
Use the annotated cell-type labels as targets for supervised machine learning classification.


## Classification
predict cell type from gene expression (via PCA features)

Features → adata.obsm["X_pca"] (50 PCs)

Labels → adata.obs["cell_type"]


## Supervised Classification of Cell Types

### Objective
Train a machine learning model to predict annotated cell types using PCA-derived features.

### Method

- Features: PCA coordinates (`X_pca`)
- Labels: annotated cell types (`cell_type`)
- Model: Random Forest classifier
- Data split: 80% training, 20% testing (stratified)

### Evaluation

Model performance was evaluated using:

- Accuracy
- Classification report (precision, recall, F1-score)
- Confusion matrix

### Interpretation

The classifier achieved strong performance, indicating that PCA features effectively capture biologically meaningful differences between cell types.

Some confusion was observed between related immune populations (e.g., cytotoxic lymphocytes and NK cells), which is expected due to overlapping gene expression profiles.

Rare cell populations (e.g., platelets, cycling cells) showed lower performance due to class imbalance.

### Conclusion

This step demonstrates how unsupervised clustering and biological annotation can be used to construct a supervised learning task, bridging exploratory data analysis and predictive modeling.

#Load data

In [ ]:
#load data
import numpy as np
import pandas as pd

X = adata.obsm["X_pca"]
y = adata.obs["cell_type"]

#Train test split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

#Train model (Random Forest)

In [ ]:
#Random forest
from sklearn.ensemble import RandomForestClassifier

clf_RF = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

clf_RF.fit(X_train, y_train)

#Prediction

In [ ]:
y_pred = clf_RF.predict(X_test)

#Evaluation

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred, labels=clf_RF.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=clf_RF.classes_)

disp.plot(xticks_rotation=90)
plt.show()

In [ ]:
importances = clf_RF.feature_importances_

plt.plot(importances)
plt.title("Feature Importance (PCA components)")
plt.xlabel("PC index")
plt.ylabel("Importance")
plt.show()

### Results Interpretation

The Random Forest classifier achieved high accuracy (~97%), indicating that PCA-derived features effectively capture biologically meaningful variation across cell types.

Most major cell populations (T cells, B cells, NK cells, and dendritic cells) were classified with near-perfect performance, reflecting strong transcriptional separation.

Mild confusion was observed between related cell types, particularly between monocyte subtypes, which is expected due to overlapping gene expression profiles.

Interestingly, cytotoxic lymphocytes and NK cells were well separated by the model, suggesting that the clustering and feature representation captured subtle biological differences between these populations.

Performance on rare cell types (platelets and cycling cells) was less stable due to very small sample sizes.

Feature importance analysis showed that the classifier relied heavily on the first few principal components, which capture the dominant biological structure of the dataset. Later components contributed less, consistent with the PCA variance distribution.

#Map feature importance → genes (via PCA loadings)

In [ ]:
import numpy as np
import pandas as pd

# PCA loadings (genes × PCs)
loadings = adata.varm["PCs"]   # shape: (n_genes, n_pcs)

# RF importance (per PC)
pc_importance = clf_RF.feature_importances_

# Combine them
gene_importance = np.dot(loadings, pc_importance)

# Convert to DataFrame
gene_importance_df = pd.DataFrame({
    "gene": adata.var_names,
    "importance": gene_importance
})

# Sort
gene_importance_df = gene_importance_df.sort_values(by="importance", ascending=False)

gene_importance_df.head(20)

#Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(
    max_iter=1000
)

lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred_lr))

#XGBoost

In [ ]:
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

# Encode string labels to integers
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print(mapping)

# Train XGBoost
xgb = XGBClassifier(
    n_estimators=100,
    random_state=42,
    eval_metric="mlogloss"
)

xgb.fit(X_train, y_train_enc)

# Predict
y_pred_xgb_enc = xgb.predict(X_test)

# Convert predictions back to original cell-type labels
y_pred_xgb = le.inverse_transform(y_pred_xgb_enc)

# Evaluate
print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print(classification_report(y_test, y_pred_xgb))

#Interpretation

All three models perform similarly → the representation (PCA) is doing the heavy lifting.

High performance across models suggests robust, well-separated cell populations.

Comparable performance between Logistic Regression and Random Forest suggests that the PCA representation effectively separates cell types in a near-linear manner.

Cell types are highly separable in PCA space, and both linear and non-linear models achieve comparable performance, indicating that the dominant biological variation is well captured by principal components.”

### Model Comparison

Three models were evaluated: Random Forest, Logistic Regression, and XGBoost. All models achieved high accuracy (~97–98%), indicating that PCA features effectively capture biologically meaningful variation across cell types.

Comparable performance between linear (Logistic Regression) and non-linear models (Random Forest, XGBoost) suggests that cell types are largely linearly separable in PCA space.

Minor performance differences were observed for closely related cell types (e.g., monocyte subtypes) and rare populations (e.g., dendritic cells, platelets), reflecting biological similarity and class imbalance.

### Feature Importance and Biological Interpretation

Feature importance from the Random Forest model was mapped back to gene space using PCA loadings.

The top-ranked genes included well-known immune markers such as:

- Cytotoxic markers: `NKG7`, `GZMA`, `GZMB`, `PRF1`, `GNLY`, `CST7`
- Chemokines: `CCL5`, `CCL4`
- Monocyte markers: `FCGR3A`, `TYROBP`, `FCER1G`
- Antigen presentation genes: `HLA-DPA1`

Even though, T cells are the largest class, top genes are mostly: cytotoxic and immune markers. Bbecause those genes provide stronger separation, not because of class size. Therefore, importance reflects discriminative power, not frequency.

These results demonstrate that the model relies on biologically meaningful genes to distinguish cell types, linking machine learning outputs directly to known immunological signals.

### Conclusion

This analysis demonstrates an end-to-end workflow combining unsupervised clustering, biological annotation, and supervised machine learning. The strong performance and interpretable feature importance highlight the effectiveness of PCA-based representations for modeling single-cell RNA-seq data.